In [ ]:
#| default_exp network

# Network

> VPC/Security Groups, Secrets Manager, IAM roles, VPC Endpoints, CloudFront, and ALB.

In [ ]:
#| export
import json

## VPC and Security Groups

Equivalent to Azure VNet/NSG.

```python
vpc = create_vpc(auth, 'my-vpc')
subnet = add_subnet(auth, vpc['VpcId'], '10.0.1.0/24', az='us-east-1a')
sg = create_security_group(auth, 'web-sg', vpc['VpcId'])
sg_rule(auth, sg['GroupId'], 'ingress', 'tcp', 443)
```

In [ ]:
#| export
_tags = lambda d: [{'Key': k, 'Value': v} for k, v in (d or {}).items()]

def _ec2(auth):
    return auth.session.client('ec2')

def create_vpc(auth, name, cidr='10.0.0.0/16', tags=None) -> dict:
    'Create a VPC with DNS resolution and hostnames enabled. Idempotent — returns existing VPC if Name tag matches.'
    ec2 = _ec2(auth)
    existing = ec2.describe_vpcs(
        Filters=[{'Name': 'tag:Name', 'Values': [name]}])['Vpcs']
    if existing:
        return existing[0]
    tag_list = [{'Key': 'Name', 'Value': name}] + _tags(tags)
    vpc = ec2.create_vpc(
        CidrBlock=cidr,
        TagSpecifications=[{'ResourceType': 'vpc', 'Tags': tag_list}],
    )['Vpc']
    ec2.modify_vpc_attribute(VpcId=vpc['VpcId'],
                             EnableDnsSupport={'Value': True})
    ec2.modify_vpc_attribute(VpcId=vpc['VpcId'],
                             EnableDnsHostnames={'Value': True})
    return vpc

def add_subnet(auth, vpc_id, cidr, az, public=False, name=None) -> dict:
    'Add a subnet to a VPC. Idempotent — returns existing subnet if CIDR+VPC matches.'
    ec2 = _ec2(auth)
    existing = ec2.describe_subnets(
        Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]},
                 {'Name': 'cidrBlock', 'Values': [cidr]}])['Subnets']
    if existing:
        return existing[0]
    tags = [{'Key': 'Name', 'Value': name or cidr}]
    subnet = ec2.create_subnet(
        VpcId=vpc_id, CidrBlock=cidr, AvailabilityZone=az,
        TagSpecifications=[{'ResourceType': 'subnet', 'Tags': tags}],
    )['Subnet']
    if public:
        ec2.modify_subnet_attribute(
            SubnetId=subnet['SubnetId'],
            MapPublicIpOnLaunch={'Value': True})
    return subnet

def create_security_group(auth, name, vpc_id, description='', tags=None) -> dict:
    'Create a security group in a VPC. Idempotent — returns existing group if name+VPC matches.'
    ec2 = _ec2(auth)
    try:
        sg = ec2.create_security_group(
            GroupName=name, Description=description or name, VpcId=vpc_id,
            TagSpecifications=[{'ResourceType': 'security-group',
                                'Tags': [{'Key': 'Name', 'Value': name}] + _tags(tags)}],
        )
        return ec2.describe_security_groups(
            GroupIds=[sg['GroupId']])['SecurityGroups'][0]
    except ec2.exceptions.ClientError as e:
        if e.response['Error']['Code'] == 'InvalidGroup.Duplicate':
            from fastcore.basics import first
            return first(ec2.describe_security_groups(
                Filters=[{'Name': 'group-name', 'Values': [name]},
                         {'Name': 'vpc-id', 'Values': [vpc_id]}])['SecurityGroups'])
        raise

def sg_rule(auth, sg_id, direction, protocol, port, cidr='0.0.0.0/0'):
    'Add an inbound or outbound rule to a security group. Idempotent — silently ignores duplicate rules.'
    ec2 = _ec2(auth)
    rule = [{'IpProtocol': protocol,
              'FromPort': port, 'ToPort': port,
              'IpRanges': [{'CidrIp': cidr}]}]
    try:
        if direction == 'ingress':
            ec2.authorize_security_group_ingress(GroupId=sg_id, IpPermissions=rule)
        else:
            ec2.authorize_security_group_egress(GroupId=sg_id, IpPermissions=rule)
    except ec2.exceptions.ClientError as e:
        if e.response['Error']['Code'] != 'InvalidPermission.Duplicate':
            raise


## AWS Secrets Manager

Equivalent to Azure Key Vault. KMS-encrypted, create-or-update semantics.

```python
create_secret(auth, 'prod/db-password', 'mysecret')
print(get_secret(auth, 'prod/db-password'))
```

In [ ]:
#| export
def _sm(auth):
    return auth.session.client('secretsmanager')

def create_secret(auth, name, value, kms_key_id=None, tags=None, **_) -> dict:
    'Create or update a Secrets Manager secret. KMS-encrypted when kms_key_id provided.'
    client = _sm(auth)
    kwargs = {'Name': name, 'SecretString': value, 'Tags': _tags(tags)}
    if kms_key_id: kwargs['KmsKeyId'] = kms_key_id
    try:
        return client.create_secret(**kwargs)
    except client.exceptions.ResourceExistsException:
        client.put_secret_value(SecretId=name, SecretString=value)
        return client.describe_secret(SecretId=name)

def get_secret(auth, name) -> str:
    'Return the secret string value.'
    return _sm(auth).get_secret_value(SecretId=name)['SecretString']

def update_secret(auth, name, value):
    'Update an existing secret value.'
    _sm(auth).put_secret_value(SecretId=name, SecretString=value)

def secret_arn(auth, name) -> str:
    'Return the ARN of a secret.'
    return _sm(auth).describe_secret(SecretId=name)['ARN']


## IAM Roles

Equivalent to Azure Managed Identity + RBAC. Service-linked or workload trust policies.

```python
r = create_role(auth, 'my-app-role', service='lambda.amazonaws.com')
attach_policy(auth, 'my-app-role', 'arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess')
print(role_arn(auth, 'my-app-role'))
```

In [ ]:
#| export
def _iam(auth):
    return auth.session.client('iam')

def create_role(auth, name, service=None, trust_policy=None, tags=None) -> dict:
    'Create an IAM role. Provide service (e.g. "ec2.amazonaws.com") or a full trust_policy dict. Idempotent.'
    client = _iam(auth)
    if trust_policy is None:
        trust_policy = {'Version': '2012-10-17', 'Statement': [{
            'Effect': 'Allow',
            'Principal': {'Service': service},
            'Action': 'sts:AssumeRole'}]}
    try:
        return client.create_role(
            RoleName=name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Tags=_tags(tags),
        )
    except client.exceptions.EntityAlreadyExistsException:
        return {'Role': client.get_role(RoleName=name)['Role']}

def attach_policy(auth, role_name, policy_arn):
    'Attach a managed IAM policy to a role.'
    _iam(auth).attach_role_policy(RoleName=role_name, PolicyArn=policy_arn)

def role_arn(auth, role_name) -> str:
    'Return the ARN of an IAM role.'
    return _iam(auth).get_role(RoleName=role_name)['Role']['Arn']


## VPC Endpoints

Equivalent to Azure Private Endpoints. Eliminates public internet exposure for AWS services.

```python
create_vpc_endpoint(auth, vpc_id, 'com.amazonaws.us-east-1.s3',
                    endpoint_type='Gateway')
```

In [ ]:
#| export
def create_vpc_endpoint(auth, vpc_id, service_name, endpoint_type='Interface',
                        subnet_ids=None, sg_ids=None) -> dict:
    'Create a VPC Endpoint (Interface or Gateway) for an AWS service.'
    kwargs = dict(
        VpcId=vpc_id,
        ServiceName=service_name,
        VpcEndpointType=endpoint_type,
    )
    if endpoint_type == 'Interface':
        if subnet_ids: kwargs['SubnetIds'] = subnet_ids
        if sg_ids:     kwargs['SecurityGroupIds'] = sg_ids
        kwargs['PrivateDnsEnabled'] = True
    return _ec2(auth).create_vpc_endpoint(**kwargs)['VpcEndpoint']

## CloudFront and ALB

Equivalent to Azure Front Door and App Gateway. CloudFront for global CDN + WAF, ALB for regional load balancing.

```python
create_distribution(auth, 'my-cdn', origin_domain='api.example.com')
create_alb(auth, 'my-alb', vpc_id=vpc_id, subnet_ids=[s1, s2])
```

In [ ]:
#| export
import time

def _cf(auth):
    return auth.session.client('cloudfront')

def _elbv2(auth):
    return auth.session.client('elbv2')

def create_distribution(auth, name, origin_domain, waf_acl_arn=None,
                        tags=None) -> dict:
    'Create a CloudFront distribution with HTTPS-only origin and TLS 1.2. Idempotent — checks for existing distribution by Comment before creating.'
    from fastcore.basics import first
    cf = _cf(auth)
    # Check for existing distribution with this name
    dists = cf.list_distributions()['DistributionList'].get('Items', [])
    existing = first((d for d in dists if d.get('Comment') == name), None)
    if existing:
        return existing

    tag_list = _tags(tags)
    dist_config = {
        'CallerReference': f'awseasy-{name}',
        'Comment': name,
        'Enabled': True,
        'Origins': {'Quantity': 1, 'Items': [{
            'Id': 'origin-1',
            'DomainName': origin_domain,
            'CustomOriginConfig': {
                'HTTPSPort': 443,
                'OriginProtocolPolicy': 'https-only',
                'OriginSSLProtocols': {'Quantity': 1, 'Items': ['TLSv1.2']},
            },
        }]},
        'DefaultCacheBehavior': {
            'TargetOriginId': 'origin-1',
            'ViewerProtocolPolicy': 'redirect-to-https',
            'CachePolicyId': '4135ea2d-6df8-44a3-9df3-4b5a84be39ad',  # CachingDisabled
            'AllowedMethods': {'Quantity': 2, 'Items': ['GET', 'HEAD'],
                               'CachedMethods': {'Quantity': 2,
                                                 'Items': ['GET', 'HEAD']}},
        },
        'ViewerCertificate': {'CloudFrontDefaultCertificate': True},
    }
    if waf_acl_arn:
        dist_config['WebACLId'] = waf_acl_arn
    return cf.create_distribution_with_tags(
        DistributionConfigWithTags={
            'DistributionConfig': dist_config,
            'Tags': {'Items': tag_list},
        })['Distribution']

def create_alb(auth, name, vpc_id, subnet_ids, scheme='internet-facing',
               waf_acl_arn=None, tags=None) -> dict:
    'Create an Application Load Balancer. Idempotent — returns existing ALB if name matches.'
    from fastcore.basics import first
    client = _elbv2(auth)
    try:
        alb = client.create_load_balancer(
            Name=name,
            Subnets=subnet_ids,
            Scheme=scheme,
            Type='application',
            IpAddressType='ipv4',
            Tags=_tags(tags),
        )['LoadBalancers'][0]
    except client.exceptions.DuplicateLoadBalancerNameException:
        alb = first(client.describe_load_balancers(
            Names=[name])['LoadBalancers'])
    if waf_acl_arn:
        auth.session.client('wafv2').associate_web_acl(
            WebACLArn=waf_acl_arn, ResourceArn=alb['LoadBalancerArn'])
    return alb
